In [1]:
import json
import shutil
from pathlib import Path

DATA_DIR = Path('data')
LOGS_ORIGINAL = DATA_DIR / 'logs_original'
LOGS_REPAIRED = DATA_DIR / 'logs'

LOGGING_FREQUENCY = 25

# Repair the last train_loss_logfreq entry of each epoch.
#
# Bug: the training loop always divides logging_loss by logging_frequency (25),
# even when the last window of an epoch has fewer than 25 batches.
#
#   logs["train_loss_logfreq"].append(logging_loss / logging_frequency)
#
# The correct divisor for the last window is: num_train_batch % logging_frequency
# (or logging_frequency if it divides evenly).
#
# To fix, we scale the affected entries: value * (25 / actual_batches)

def repair_logs(data):
    data = json.loads(json.dumps(data))  # deep copy

    n_log_steps = len(data['train_loss_logfreq'])
    n_epochs = len(data['train_loss_epoch'])
    steps_per_epoch = n_log_steps // n_epochs

    # Total batches per epoch = steps from modulo + possible remainder
    # steps_per_epoch log points: (steps_per_epoch - 1) from modulo, 1 from end-of-epoch
    # OR all from modulo if num_train_batch is exactly divisible
    #
    # num_train_batch = (steps_per_epoch - 1) * logging_frequency + remainder
    # where remainder is the actual batch count in the last window.
    #
    # We can recover the remainder from the data:
    # If remainder == logging_frequency, no correction needed.
    # If remainder < logging_frequency, the logged value = true_avg * (remainder / logging_frequency)
    # So: true_avg = logged_value * (logging_frequency / remainder)
    #
    # We can figure out remainder by checking if the last point is an outlier.
    # But more reliably: epoch_loss = sum of all batch losses / num_train_batch
    # And we know all the logged window averages.
    #
    # Approach: use epoch-level loss to solve for the correct last-window average.
    # epoch_loss_reported = epoch_loss_total / num_train_batch
    # epoch_loss_total = sum over all windows of (window_avg * window_size)
    # For all windows except the last: window_size = logging_frequency
    # For the last window: window_size = remainder
    #
    # epoch_loss_total = sum(first (n-1) windows * 25) + last_window_true_avg * remainder
    # epoch_loss_reported = epoch_loss_total / num_train_batch
    # num_train_batch = (steps_per_epoch - 1) * 25 + remainder
    #
    # We have two unknowns (last_window_true_avg, remainder) but we know:
    #   logged_last = last_window_true_avg * remainder / 25
    # So: last_window_true_avg = logged_last * 25 / remainder
    #
    # Substituting into the epoch loss equation:
    # epoch_loss_reported * num_train_batch = sum_first * 25 + (logged_last * 25 / remainder) * remainder
    # epoch_loss_reported * num_train_batch = sum_first * 25 + logged_last * 25
    # epoch_loss_reported * ((n-1)*25 + remainder) = (sum_first + logged_last) * 25
    # remainder = (sum_first + logged_last) * 25 / epoch_loss_reported - (n-1) * 25

    repaired_count = 0

    for epoch_idx in range(n_epochs):
        start = epoch_idx * steps_per_epoch
        end = (epoch_idx + 1) * steps_per_epoch
        window_values = data['train_loss_logfreq'][start:end]
        epoch_loss = data['train_loss_epoch'][epoch_idx]

        sum_all_windows = sum(window_values)

        # Solve for remainder
        n_full_windows = steps_per_epoch - 1
        remainder = (sum_all_windows * LOGGING_FREQUENCY) / epoch_loss - n_full_windows * LOGGING_FREQUENCY

        remainder_rounded = round(remainder)

        if remainder_rounded != LOGGING_FREQUENCY and remainder_rounded > 0:
            # The last entry needs correction
            correction_factor = LOGGING_FREQUENCY / remainder_rounded
            old_val = data['train_loss_logfreq'][end - 1]
            new_val = old_val * correction_factor
            data['train_loss_logfreq'][end - 1] = new_val
            repaired_count += 1

            if epoch_idx == 0:
                print(f"  Detected {remainder_rounded} batches in last window (not {LOGGING_FREQUENCY})")
                print(f"  Correction factor: {correction_factor:.4f}")

            print(f"  Epoch {epoch_idx + 1}: {old_val:.4f} → {new_val:.4f}")

    return data, repaired_count

# Process all experiments
LOGS_REPAIRED.mkdir(exist_ok=True)

for exp_dir in sorted(LOGS_ORIGINAL.iterdir()):
    if not exp_dir.is_dir():
        continue

    log_file = exp_dir / 'args.json'
    if not log_file.exists():
        continue

    with open(log_file) as f:
        original = json.load(f)

    print(f"\n=== Experiment {exp_dir.name} ===")
    repaired, count = repair_logs(original)

    # Write repaired log
    out_dir = LOGS_REPAIRED / exp_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / 'args.json', 'w') as f:
        json.dump(repaired, f, indent=2)

    print(f"  Repaired {count} entries, saved to {out_dir / 'args.json'}")

print("\nDone.")


=== Experiment 1 ===
  Detected 17 batches in last window (not 25)
  Correction factor: 1.4706
  Epoch 1: 0.1577 → 0.2319
  Epoch 2: 0.1403 → 0.2063
  Epoch 3: 0.1286 → 0.1890
  Epoch 4: 0.1021 → 0.1502
  Epoch 5: 0.0968 → 0.1424
  Repaired 5 entries, saved to data/logs/1/args.json

=== Experiment 2 ===
  Detected 17 batches in last window (not 25)
  Correction factor: 1.4706
  Epoch 1: 0.1965 → 0.2890
  Epoch 2: 0.1681 → 0.2472
  Epoch 3: 0.1437 → 0.2113
  Epoch 4: 0.1353 → 0.1989
  Epoch 5: 0.1294 → 0.1903
  Repaired 5 entries, saved to data/logs/2/args.json

=== Experiment 3 ===
  Detected 17 batches in last window (not 25)
  Correction factor: 1.4706
  Epoch 1: 0.1891 → 0.2781
  Epoch 2: 0.1548 → 0.2277
  Epoch 3: 0.1476 → 0.2171
  Epoch 4: 0.1317 → 0.1937
  Epoch 5: 0.1232 → 0.1812
  Repaired 5 entries, saved to data/logs/3/args.json

=== Experiment 4 ===
  Detected 17 batches in last window (not 25)
  Correction factor: 1.4706
  Epoch 1: 0.1910 → 0.2809
  Epoch 2: 0.1540 → 0.226